# Summary statistic diagnostics for NLE-friendliness

Checks whether learned VMIM summaries are well-conditioned inputs for the downstream NLE flow,
comparing regularized (VICReg variance+covariance) against unregularized runs.

**Gate 2 criteria** (see plan / `configs/loss/vmim_vicreg_var_cov.yaml`):
- per-dimension std ≈ 1 (no collapsed, near-constant dimensions)
- low off-diagonal correlation (no duplicated dimensions)
- no near-zero eigenvalues in the summary covariance; effective rank close to `n_summary`

Reads the `preds_*.h5` files written by `deep_lss/apps/run_cls_training+evaluation.py`.

In [ ]:
import glob, os

import h5py
import matplotlib.pyplot as plt
import numpy as np

runs_dir = "/users/athomsen/scratch/deep_lss/runs"

# model_name -> pred_dir, as passed to run_cls_training+evaluation.py via --out_dir/--model_name
# EDIT: point these at the 2x2 ablation runs
run_dirs = {
    "vmim_fac2": os.path.join(runs_dir, "cls_ablation/vmim"),
    "vicreg_fac2": os.path.join(runs_dir, "cls_ablation/vmim_vicreg_var_cov"),
    "vmim_fac1": os.path.join(runs_dir, "cls_ablation/vmim_fac1"),
    "vicreg_fac1": os.path.join(runs_dir, "cls_ablation/vmim_vicreg_var_cov_fac1"),
}

In [ ]:
def load_preds(pred_dir):
    pred_files = sorted(glob.glob(os.path.join(pred_dir, "preds_*.h5")))
    assert len(pred_files) > 0, f"no preds_*.h5 in {pred_dir}"
    with h5py.File(pred_files[-1], "r") as f:
        preds = f["grid/preds/test"][:]
        cosmos = f["grid/cosmos/test"][:]
    return preds, cosmos


def effective_rank(eigvals, eps=1e-12):
    """exp of the entropy of the normalized eigenvalue spectrum (Roy & Vetterli 2007)"""
    p = eigvals / (eigvals.sum() + eps)
    return float(np.exp(-np.sum(p * np.log(p + eps))))


summaries = {}
for name, pred_dir in run_dirs.items():
    try:
        preds, cosmos = load_preds(pred_dir)
        summaries[name] = {"preds": preds, "cosmos": cosmos}
        print(f"{name:>15s}: preds {preds.shape}, cosmos {cosmos.shape}")
    except (AssertionError, OSError) as e:
        print(f"{name:>15s}: skipped ({e})")

## Per-dimension mean and std

Collapsed dimensions show up as std ≪ 1. With the VICReg variance term active, all dimensions
should sit close to std = 1; the unregularized runs are free to drift.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, d in summaries.items():
    s = d["preds"]
    axes[0].plot(s.mean(axis=0), "o-", label=name)
    axes[1].semilogy(s.std(axis=0), "o-", label=name)
axes[0].set_ylabel("mean")
axes[1].set_ylabel("std")
axes[1].axhline(1.0, color="k", ls=":", lw=1)
for ax in axes:
    ax.set_xlabel("summary dimension")
    ax.legend(fontsize=8)
fig.tight_layout()

for name, d in summaries.items():
    std = d["preds"].std(axis=0)
    print(f"{name:>15s}: std min/max = {std.min():.3e} / {std.max():.3e}")

## Correlation matrices

Duplicated/redundant dimensions show up as |corr| ≈ 1 off the diagonal.

In [ ]:
n_runs = len(summaries)
fig, axes = plt.subplots(1, n_runs, figsize=(4 * n_runs, 3.5), squeeze=False)
for ax, (name, d) in zip(axes[0], summaries.items()):
    corr = np.corrcoef(d["preds"], rowvar=False)
    im = ax.imshow(corr, vmin=-1, vmax=1, cmap="RdBu_r")
    ax.set_title(name, fontsize=10)
    plt.colorbar(im, ax=ax, fraction=0.046)
    off_diag = corr[~np.eye(len(corr), dtype=bool)]
    print(f"{name:>15s}: max |off-diag corr| = {np.abs(off_diag).max():.3f}")
fig.tight_layout()

## Covariance eigenspectrum and effective rank

Near-zero eigenvalues are what destabilize the NLE flow (near-singular $p(s\\,|\\,\\theta)$).
The effective rank tells you how many dimensions VMIM actually uses — this informs the
`dim_summary_fac` choice: if the fac=2 run has effective rank ≈ n_params, the extra
dimensions carry no information (harmless when regularized, pathological when not).

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for name, d in summaries.items():
    s = d["preds"]
    cov = np.cov(s, rowvar=False)
    eigvals = np.sort(np.linalg.eigvalsh(cov))[::-1]
    er = effective_rank(eigvals)
    ax.semilogy(eigvals, "o-", label=f"{name} (eff. rank {er:.1f})")
    print(f"{name:>15s}: eff. rank = {er:5.2f} / {s.shape[1]}, cond. number = {eigvals[0] / eigvals[-1]:.2e}")
ax.set_xlabel("eigenvalue index")
ax.set_ylabel("eigenvalue of Cov(s)")
ax.legend(fontsize=8)
fig.tight_layout()

## Parameter dependence per dimension

$R^2$ of a linear regression of each summary dimension on the cosmological parameters.
Dimensions with $R^2 \\approx 0$ are (linearly) uninformative — expected to behave as benign
$\\mathcal{N}(0,1)$ noise dims in the regularized runs.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for name, d in summaries.items():
    s, theta = d["preds"], d["cosmos"]
    X = np.concatenate([theta, np.ones((len(theta), 1))], axis=1)
    coef, *_ = np.linalg.lstsq(X, s, rcond=None)
    resid = s - X @ coef
    r2 = 1.0 - resid.var(axis=0) / s.var(axis=0)
    ax.plot(r2, "o-", label=name)
ax.set_xlabel("summary dimension")
ax.set_ylabel("$R^2$ (linear in $\\theta$)")
ax.set_ylim(-0.05, 1.05)
ax.legend(fontsize=8)
fig.tight_layout()